# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds the monthly decision table for the **Refresh / Content Opportunity Scoring** lane. One row is one pseudonymized content item at one feature-month anchor. The outcome asks whether measured GSC impressions in the next consecutive month fall below 80% of the feature month. This is a directional decision-support label, not proof that a refresh causes movement.

## 1. Build the feature vector

The remote daily warehouse was aggregated once into an ignored page-month cache. Current-month and prior-month fields are available before the recommendation; the next month is retained only for the label. Eligible examples have at least 100 current impressions and at least 20 measured GSC days in both the feature and outcome months.

In [1]:
from pathlib import Path
import json
import sys

import duckdb
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'work' / 'scripts').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / 'work' / 'scripts').exists():
    raise RuntimeError('Run this notebook from the repository or a child directory')
sys.path.insert(0, str(ROOT))

from work.scripts.build_monthly_features import (
    FEATURE_COLUMNS, add_derived_features, feature_contract_records, validate_feature_contract,
)
from work.scripts.refresh_capstone import make_examples, split_frames

cache_path = ROOT / 'work' / 'outputs' / 'page_month_features.parquet'
if not cache_path.exists():
    raise FileNotFoundError('Build the ignored page-month cache with work.scripts.build_monthly_features first')
con = duckdb.connect()
monthly = con.execute('SELECT * FROM read_parquet(?)', [str(cache_path)]).df()
monthly['client_hash_id'] = monthly['client_hash_id'].astype('category')
monthly['content_hash_id'] = monthly['content_hash_id'].astype('category')
examples = add_derived_features(make_examples(monthly))
examples = examples.loc[pd.to_datetime(examples['month']).between('2025-09-01', '2026-05-01')].copy()
validate_feature_contract(examples, FEATURE_COLUMNS)
examples_path = ROOT / 'work' / 'outputs' / 'model_examples.parquet'
con.register('examples_frame', examples)
con.execute('COPY examples_frame TO ? (FORMAT PARQUET, COMPRESSION ZSTD)', [str(examples_path)])
train, validation, sealed = split_frames(examples)
summary = pd.DataFrame({
    'split': ['training', 'validation', 'sealed'],
    'rows': [len(train), len(validation), len(sealed)],
    'clients': [x['client_hash_id'].nunique() for x in (train, validation, sealed)],
    'displayed_label_rate': [f'{train["future_decline"].mean():.3f}', f'{validation["future_decline"].mean():.3f}', 'sealed—not opened'],
})
print(f'Page-month cache: {len(monthly):,} rows; duplicate decision-grain rows: {monthly.duplicated(["month", "client_hash_id", "content_hash_id"]).sum():,}')
print(f'Eligible labeled examples: {len(examples):,}')
summary

Page-month cache: 2,782,172 rows; duplicate decision-grain rows: 0
Eligible labeled examples: 528,068


,split,rows,clients,displayed_label_rate
0,training,341546,41,0.293
1,validation,93474,43,0.538
2,sealed,93048,44,sealed—not opened


## 2. Feature notes: meaning, missingness, and availability

Missing measurement remains missing. In particular, GA4 engaged sessions are not filled with zero when no GA4 day is available, and position is accompanied by an explicit validity flag. Numeric imputation, if used by a later model, will be fit on training rows only and paired with missingness indicators. IDs remain context fields and will never be predictive inputs.

In [2]:
contract_records = feature_contract_records()
contract = {
    'warehouse_release': 'flyrank_pseudonymized_warehouse_release_v20260703',
    'unit_of_analysis': 'one pseudonymized content item at one monthly decision anchor',
    'feature_window': 'month t plus prior consecutive month where available',
    'label_window': 'next consecutive month t+1',
    'eligibility': 'at least 100 feature-month impressions and 20 measured GSC days in t and t+1',
    'feature_columns': list(FEATURE_COLUMNS),
    'fields': contract_records,
}
contract_path = ROOT / 'work' / 'outputs' / 'feature_contract.json'
contract_path.write_text(json.dumps(contract, indent=2) + '\n', encoding='utf-8')
feature_notes = pd.DataFrame([row for row in contract_records if row['bucket'] == 'feature'])
missingness = examples[list(FEATURE_COLUMNS)].isna().mean().rename('missing_share').to_frame()
display(feature_notes)
display(missingness.sort_values('missing_share', ascending=False))
print('GA4 unavailable rows retain missing engagement:', bool(examples.loc[examples['ga4_available_days'].eq(0), 'engaged_sessions'].isna().all()))

,field,bucket,source_window,availability_rule
0,impressions,feature,feature month,measured GSC exposure; eligible rows require a...
1,clicks,feature,feature month,measured GSC clicks; not filled across unavail...
2,ctr,feature,feature month,clicks divided by impressions; missing when de...
3,prior_impressions,feature,prior month,previous measured exposure; missing without a ...
4,prior_clicks,feature,prior month,previous measured clicks; missing without a co...
5,impression_momentum,feature,feature/prior month,current divided by prior impressions; missing ...
6,avg_position,feature,feature month,impression-weighted valid GSC position only
7,position_available,feature,feature month,explicit flag for a valid position denominator
8,active_days,feature,feature month,days with measured positive GSC impressions
9,gsc_available_days,feature,feature month,measurement coverage; labels require 20 days i...


,missing_share
engaged_sessions,0.561072
impression_momentum,0.049791
prior_impressions,0.024224
prior_clicks,0.024224
search_volume,0.019143
avg_position,0.000008
ctr,0.000000
clicks,0.000000
impressions,0.000000
position_available,0.000000


GA4 unavailable rows retain missing engagement: True


## 3. The leakage hunt

I attack the feature list directly rather than relying on naming intuition. The assertions below reject target/outcome fields, identifiers, query/URL-like fields, non-consecutive windows, labels without enough measured coverage, and anchors outside the declared split. June labels remain sealed for later one-time evaluation.

In [3]:
validate_feature_contract(examples, FEATURE_COLUMNS)
feature_month = pd.to_datetime(examples['month']).dt.to_period('M')
outcome_month = pd.to_datetime(examples['outcome_month']).dt.to_period('M')
assert (outcome_month == feature_month + 1).all()
assert examples['gsc_available_days'].ge(20).all()
assert examples['outcome_gsc_available_days'].ge(20).all()
assert examples['impressions'].ge(100).all()
assert feature_month.min() >= pd.Period('2025-09', freq='M')
assert feature_month.max() <= pd.Period('2026-05', freq='M')
assert outcome_month.max() <= pd.Period('2026-06', freq='M')
assert not any(name.startswith(('outcome_', 'future_')) for name in FEATURE_COLUMNS)
assert not {'client_hash_id', 'content_hash_id', 'month'}.intersection(FEATURE_COLUMNS)
development_examples = examples.loc[feature_month <= pd.Period('2026-04', freq='M')]
anchor_check = development_examples.groupby('month', observed=True).agg(
    rows=('future_decline', 'size'),
    clients=('client_hash_id', 'nunique'),
    positive_rate=('future_decline', 'mean'),
).reset_index()
print(f'Sealed May anchor coverage: {len(sealed):,} rows across {sealed["client_hash_id"].nunique():,} clients; labels not summarized or used.')
print('Leakage assertions passed. Feature information ends before each outcome month begins.')
anchor_check

Sealed May anchor coverage: 93,048 rows across 44 clients; labels not summarized or used.
Leakage assertions passed. Feature information ends before each outcome month begins.


,month,rows,clients,positive_rate
0,2025-09-01,18684,15,0.268893
1,2025-10-01,26012,18,0.161771
2,2025-11-01,35384,25,0.254720
3,2025-12-01,49277,26,0.215598
4,2026-01-01,58778,27,0.237572
5,2026-02-01,64470,24,0.180890
6,2026-03-01,88941,37,0.512542
7,2026-04-01,93474,43,0.537946


## 4. What I excluded and why

I excluded all future/outcome measurements because they answer the prediction question; pseudonymous IDs because identity is non-portable; raw names, domains, URLs, queries, and titles because they are private or identifying; provider/model fields because they do not support the editorial decision; and the fixed 90-day query table because its window can overlap final outcomes. The model therefore estimates observed directional risk from pre-decision, public-safe measurements only.

In [4]:
excluded = pd.DataFrame([row for row in contract_records if row['bucket'] == 'excluded'])
prohibited_public_columns = {'client_name', 'domain', 'url', 'query', 'title', 'credentials'}
assert prohibited_public_columns.isdisjoint({column.lower() for column in examples.columns})
assert set(pd.DataFrame(contract_records)['bucket']) == {'feature', 'label', 'context', 'excluded'}
display(excluded)
print('Privacy check passed: no client names, domains, URLs, queries, titles, or credentials are present.')

,field,bucket,source_window,availability_rule
0,raw names/domains/URLs/queries/titles,excluded,not used,private or identifying
1,all outcome-month fields,excluded,future,would leak the answer
2,IDs as model features,excluded,stable pseudonym,non-portable identity signal
3,provider_used/model_used,excluded,content metadata,not needed for the editorial decision
4,fact_content_query_90d,excluded,fixed rolling window,overlaps final outcomes and creates leakage risk


Privacy check passed: no client names, domains, URLs, queries, titles, or credentials are present.


## Self-check

- [x] Every section contains reasoning and supporting code.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, domains, URLs, private queries, or credentials appear.
- [x] Claims are limited to observed, measured, directional decision support.
- [x] Features end before the outcome window and the sealed month is not used for selection.